# 1.3 AirSim Drone Basic Control

# 1.3 AirSim Drone Basic Control

The AirSim SDK provides rich programming interfaces for drone simulation, supporting basic drone control, sensor data acquisition, and environment interaction through code. Here is a brief description of its core features:

### 1. **Connection and Initialization**
   - **Simulator Connection**: Establish connection with the AirSim simulator via the `MultirotorClient` class, confirm communication status, then take control.
   - **Unlock and Start**: Use `enableApiControl(True)` and `armDisarm(True)` to unlock the drone and put it in operable state.

### 2. **Basic Flight Control**
   - **Takeoff and Hover**: Use `takeoffAsync().join()` to take off, `hoverAsync().join()` to maintain hover.
   - **Path Planning**: Call `moveToPositionAsync(x, y, z, speed)` to move the drone to specified coordinates, with adjustable speed parameter.
   - **Return and Reset**: Use `reset()` to reset drone position, `goHomeAsync()` to return to the starting point.

### 3. **Sensor and State Retrieval**
   - **Sensor Data**: Get real-time data from IMU, GPS, barometer and other sensors via API (e.g., `getImuData()`, `getGpsData()`).
   - **State Monitoring**: `getMultirotorState()` returns real-time position, velocity, attitude and other state information.

### 4. **Advanced Features**
   - **Multi-vehicle Coordination**: Define multiple drones via config file and control each by name in code.
   - **Environment Interaction**: Supports dynamic weather adjustment (rain/snow), lighting conditions, or controlling scene object properties via API.
   - **ROS Integration**: Provides ROS/ROS2 interface for integration with robot operating system, extending complex tasks.

### 5. **SDK Design Features**
   - **Cross-platform Support**: Compatible with Python, C++, C# and other languages, applicable to Windows, Linux, macOS.
   - **Modular Architecture**: Based on RPC communication (e.g., msgpack-rpc), decoupling simulation engine from external control logic.
   - **Physics and Visual Simulation Fusion**: Combining Unreal Engine rendering with custom physics models for high-fidelity simulation.

Through these interfaces, developers can quickly build drone control algorithm testing platforms, leveraging AirSim's realistic scenes and sensor simulation to verify visual navigation, autonomous obstacle avoidance and other complex features.

This section covers AirSim initialization and basic flight control.

In [ ]:
!pip install numpy

In [ ]:
import sys
sys.path.append('..')
import airsim
import math
import time
import numpy as np

# Create client once, reuse in all subsequent cells
client = airsim.MultirotorClient()
client.confirmConnection()

## Takeoff and Landing

In [ ]:
# Reset drone state, return to starting position
client.reset()

# Enable API control (disabled by default)
client.enableApiControl(True)

# Arm/unlock, same process as real aircraft
client.armDisarm(True)

In [ ]:
# Takeoff - many drone/car control functions have Async suffix, returning immediately without blocking
# But commands execute sequentially, so use .join() to wait for completion before the next command
# You can skip .join() but then you won't know if the previous command finished
client.takeoffAsync().join()
print("Takeoff-------------------------")

In [ ]:
# Land
client.landAsync().join()
print("Landing-------------------------")

# lock
client.armDisarm(False)

## Waypoint Flight - Offboard

In [ ]:
# Reset drone state, return to starting position
client.reset()

# Enable API control (disabled by default)
client.enableApiControl(True)

# Arm/unlock, same process as real aircraft
client.armDisarm(True)

# Takeoff - many drone/car control functions have Async suffix, returning immediately without blocking
# But commands execute sequentially, so use .join() to wait for completion before the next command
# You can skip .join() but then you won't know if the previous command finished
client.takeoffAsync().join()
print("Takeoff--------------------------------")

In [ ]:
# moveToZAsync - NED coordinate system, negative Z is up
client.moveToZAsync(-3, 1).join()   # Rise to 3m altitude, speed=1

In [ ]:
# Initialize 4 waypoints and mark them in viewport
points = [airsim.Vector3r(5, 0, -3),
          airsim.Vector3r(5, 5, -3),
          airsim.Vector3r(0, 5, -3),
          airsim.Vector3r(0, 0, -3)]
client.simPlotPoints(points, color_rgba=[0, 1, 0, 1], size=30, is_persistent=True)

In [ ]:
# Method 1: Fly to each point sequentially, forming a square
client.moveToPositionAsync(5, 0, -3, 1).join()  # Move to position (5,0,-3), speed=1
client.moveToPositionAsync(5, 5, -3, 1).join()
client.moveToPositionAsync(0, 5, -3, 1).join()
client.moveToPositionAsync(0, 0, -3, 1).join()


In [ ]:
# Method 2: Fly square trajectory using waypoints
client.moveOnPathAsync(points, 1)

In [ ]:
# Land
client.landAsync().join()
print("Landing----------------------------------------------")

# lock
client.armDisarm(False)

## Drone State

In AirSim, `simGetGroundTruthKinematics` and `simGetVehiclePose` are both interfaces for getting drone state, but **they differ fundamentally in data precision, purpose, and underlying meaning**. Here is a detailed comparison:

---

### **1. `simGetVehiclePose()`**
- **Function**: Get the drone's **Pose**, i.e., position and orientation.
- **Return Data**:
  ```python
  Pose(
      position=Vector3r(x, y, z),  # Position (NED coordinate system, unit: meters)
      orientation=Quaternionr(w, x, y, z)  # Quaternion orientation (relative to world frame)
  )
  ```
- **Features**:
  - Data comes from the simulated **sensor model** (e.g., GPS, IMU), may contain noise.
  - Simulates real drone sensor output, suitable for testing SLAM, navigation algorithms.
  - If sensor noise is enabled in AirSim settings (disabled by default), returned coordinates will have perturbation.

---

### **2. `simGetGroundTruthKinematics()`**
- **Function**: Get the drone's **true kinematic state** (Ground Truth).
- **Return Data**:
  ```python
  KinematicsState(
      position=Vector3r(x, y, z),  # Position (absolutely precise, NED coordinate system)
      orientation=Quaternionr(w, x, y, z),  # Orientation (absolutely precise)
      linear_velocity=Vector3r(vx, vy, vz),  # Linear velocity (m/s)
      angular_velocity=Vector3r(wx, wy, wz),  # Angular velocity (rad/s)
      linear_acceleration=Vector3r(ax, ay, az),  # Linear acceleration (m/s^2)
      angular_acceleration=Vector3r(ax, ay, az)  # Angular acceleration (rad/s^2)
  )
  ```
- **Features**:
  - Data comes from the simulation **physics engine**, absolutely precise "god's eye view" data with no noise.
  - Suitable for **algorithm verification** (comparing sensor data with ground truth) or **control algorithm design** (direct velocity feedback).

---

### **3. Key Differences Summary**
| Feature | `simGetVehiclePose()` | `simGetGroundTruthKinematics()` |
|---------|----------------------|-------------------------------|
| Data Source | Sensor model (may contain noise) | Physics engine (absolutely precise) |
| Data Dimensions | Position and orientation only | Position, orientation, velocity, acceleration |
| Use Case | Simulate real sensor output | Algorithm verification, closed-loop control design |
| Computational Cost | Low (only reads sensor data) | High (requires full state from physics engine) |

---

### **4. Usage Recommendations**
- **Need sensor-level data** (e.g., testing SLAM, obstacle avoidance):
  Use `simGetVehiclePose()`, and enable sensor noise (via AirSim settings file configuration).

- **Need ground truth verification** (e.g., controller debugging, trajectory tracking):
  Use `simGetGroundTruthKinematics().position` or `linear_velocity`.

- **Need dynamics parameters** (e.g., velocity feedback, acceleration feedforward):
  Directly read `linear_velocity` and `linear_acceleration`.


In [ ]:
# Reset drone state, return to starting position
client.reset()

# Enable API control (disabled by default)
client.enableApiControl(True)

# Arm/unlock, same process as real aircraft
client.armDisarm(True)

# Takeoff
client.takeoffAsync().join()
print("Takeoff--------------------------------")

In [ ]:
# moveToZAsync - NED coordinate system, negative Z is up
client.moveToZAsync(-3, 1).join()   # Rise to 3m altitude, speed=1

for i in range(2):
    kinematic_state_groundtruth = client.simGetGroundTruthKinematics(vehicle_name='')
    #print("kinematic_state_groundtruth: ", kinematic_state_groundtruth)

    # 6 properties of state_groundtruth
    print(
    "position", kinematic_state_groundtruth.position,  # Position
    "\n\n linear_velocity", kinematic_state_groundtruth.linear_velocity,  # Velocity
    "\n\n linear_acceleration", kinematic_state_groundtruth.linear_acceleration , # Acceleration
    "\n\n orientation", kinematic_state_groundtruth.orientation,  # Orientation
    "\n\n angular_velocity", kinematic_state_groundtruth.angular_velocity,  # Angular velocity
    "\n\n angular_acceleration", kinematic_state_groundtruth.angular_acceleration,  # Angular acceleration
    )

    # Drone global position ground truth
    x = kinematic_state_groundtruth.position.x_val  # X coordinate in global frame
    y = kinematic_state_groundtruth.position.y_val  # Y coordinate in global frame
    z = kinematic_state_groundtruth.position.z_val  # Z coordinate in global frame
    print("x: ", x, "y: ", y, "z: ", z)

    time.sleep(1)
    print("-----------------------------------------------------------------------------------")

In [ ]:
pose = client.simGetVehiclePose()
print(pose)
print(pose.position.x_val, pose.position.y_val, pose.position.z_val)
print(pose.orientation)
print(airsim.to_eularian_angles(pose.orientation))

In [ ]:
# Land
client.landAsync().join()
print("Landing----------------------------------------------")

# lock
client.armDisarm(False)